# Placement and protection pilot

This notebook runs a **local CPU simulation**, not real distributed training. No Ray, GPU, cloud account, Z3 or RL training is involved.

Read `README.md` and `MODEL.md` for assumptions. The simulator models versioned shards, existing replica groups, additional backups, and checkpoint rollback. Parameters are synthetic and not calibrated to GPUs.

Run the setup/settings cells, then **Run / resume**. A 30-minute limit pauses the experiment; rerun the same cell to continue. Plot cells only read saved files. To change simulation or planning settings, choose a new `RUN_NAME` so results are not mixed.


In [ ]:
from pathlib import Path
import json
import sys

# Works when Jupyter starts in the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "learnedplacement" / "runner.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the LearnedPlacement checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from learnedplacement.runner import run, status, read_json, paired_report, export_run
from learnedplacement.model import make_case
print("Repository:", ROOT)
print("Python:", sys.version.split()[0])


## Editable settings

These defaults describe 24 cases: 2 sizes × 4 regimes × 3 instances. There are seven planner variants. Every plan is scored on 256 held-out traces; its planning uses a different set of 32 traces. Failure-free behavior is checked in the correctness suite.

`MAX_MINUTES` limits each invocation and can be changed when resuming. All other changes below require a new run directory. Time-limited search may explore different numbers of candidates on different CPUs.


In [ ]:
RUN_NAME = "pilot_v1"
MAX_MINUTES = 30

CONFIG = read_json(ROOT / "configs" / "pilot.json")
CONFIG["active_nodes"] = [8, 16]
CONFIG["regimes"] = ["homogeneous", "cheap_risky", "fast_risky", "correlated"]
CONFIG["instances_per_regime"] = 3
CONFIG["planning_traces"] = 32
CONFIG["evaluation_traces"] = 256
CONFIG["greedy_candidates"] = 16
CONFIG["local_candidates"] = 64
CONFIG["greedy_seconds"] = 5.0
CONFIG["local_seconds"] = 15.0
CONFIG["workload"]["backup_budget_gib"] = 8.0
CONFIG["workload"]["backup_every"] = 20
CONFIG["workload"]["checkpoint_every"] = 80
CONFIG["workload"]["required_success"] = 0.95

OUTPUT = ROOT / "results" / RUN_NAME
print("Output:", OUTPUT)
print(json.dumps(CONFIG, indent=2))


## Inspect one generated case

No simulation runs in this cell. Machine prices, speeds, risk and bandwidth are generated in `learnedplacement/model.py`. Here you can see the actual deadlines and cost budgets before starting.


In [ ]:
example = make_case(CONFIG, CONFIG["active_nodes"][0], CONFIG["regimes"][0], 0)
print("Case:", example.case_id)
print("Candidate machines:", len(example.nodes))
print("Groups / shards:", example.replicas, "/", example.shards)
print("Deadline (s):", round(example.deadline_s, 2))
print("Expected-cost budget ($):", round(example.cost_budget_usd, 4))
print("Simulation horizon (s):", round(example.max_simulated_s, 2))
print(json.dumps([vars(n) for n in example.nodes], indent=2))


## Run / resume

This cell starts the experiment. It uses one CPU process. Completed cases/plans and batches of observations save locally. Interrupting the cell is supported. A killed kernel may leave `.run.lock`; remove that file only after confirming the old process has stopped.

The exact source/configuration fingerprint is checked on resume. If you pull changed core code or edit the configuration, use a new run name. No completed result is overwritten by a different experiment.


In [ ]:
run_status = run(config=CONFIG, output=OUTPUT, max_minutes=MAX_MINUTES)
print(json.dumps(run_status, indent=2))


## Progress and results table

You can run these cells after a pause. They only read saved results. `statistical_screen_passed` is a per-plan Monte Carlo screen, not a real-world guarantee. Censored traces have no reported mean completion time or total cost.


In [ ]:
print(json.dumps(status(OUTPUT), indent=2))

import pandas as pd
from IPython.display import display, FileLink

summary_path = OUTPUT / "summary.csv"
if summary_path.exists():
    results = pd.read_csv(summary_path)
    display(results[["case_id", "method", "mean_completion_s", "mean_cost_usd",
                     "deadline_success", "deadline_success_lower95",
                     "statistical_screen_passed", "censored_traces", "planning_s", "candidates"]])
else:
    print("No completed comparisons yet. Resume the run above.")


## Plots from saved results

Each point compares the same case against the separate placement/protection planner. Lower time at higher cost is a trade-off. Check feasibility in the returned table. Planning-time plots for `joint_local_no_backup` show only ablation construction; deploying its inherited placement also requires the parent's search time.


In [ ]:
from learnedplacement.plots import plot_results

ACTIVE_NODES_TO_PLOT = 8  # Change to 16 when those cases are available.
table = plot_results(OUTPUT, active_nodes=ACTIVE_NODES_TO_PLOT)
if table is not None:
    display(table)


## Paired comparisons

Negative time difference favors the selected planner. Bootstrap intervals describe simulation variability for that case, not uncertainty about the simulator. Both plans must be checked against the budgets; a time improvement alone is not a constrained-planning win. This does not tune or reselect plans using the evaluation data.


In [ ]:
comparisons = paired_report(OUTPUT, selected="joint_local", baseline="separate")
display(pd.DataFrame(comparisons))

# Same placement, only additional backups differ.
backup_effect = paired_report(OUTPUT, selected="joint_local", baseline="joint_local_no_backup")
display(pd.DataFrame(backup_effect))


## Export for sharing

Stop or pause the runner first. The ZIP includes all saved observations, configuration, selected plans and the source snapshot. It also works for an incomplete run. This cell does not upload anything.


In [ ]:
archive = export_run(OUTPUT)
print("Share this ZIP:", archive)
# Relative path for Jupyter's file-serving URL.
import os
display(FileLink(os.path.relpath(archive, Path.cwd())))
